In [22]:
import pandas as pd
import matplotlib.pyplot as plt

df_train = pd.read_csv('/Users/tg197682/Downloads/COMP3931_Individual_Project/student-addiction-risk-prediction/data/processed/encoded_student_addiction_dataset_train.csv')
df_test = pd.read_csv('/Users/tg197682/Downloads/COMP3931_Individual_Project/student-addiction-risk-prediction/data/processed/encoded_student_addiction_dataset_test.csv')



In [14]:
# Separating features and target variable
x_train = df_train.drop('Addiction_Class', axis=1)
y_train = df_train['Addiction_Class']



In [15]:
# Seperating features and target (for training only)

from sklearn.model_selection import train_test_split

x_tr, x_val, y_tr, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42
)


In [16]:
# baseline logistic regression model

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(x_tr, y_tr)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [17]:
# Evaluating the model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = model.predict(x_val)

print("Accuracy:", accuracy_score(y_val, y_pred))
print("Precision:", precision_score(y_val, y_pred))
print("Recall:", recall_score(y_val, y_pred))
print("F1 Score:", f1_score(y_val, y_pred))

Accuracy: 0.5096831860164862
Precision: 0.30724346076458753
Recall: 0.5054617676266137
F1 Score: 0.3821799524465023


In [18]:
# tuning logistic regression model
Cs = [0.01, 0.1, 1, 10, 100]
results = []
for C in Cs:
    model = LogisticRegression(class_weight='balanced', C=C, max_iter=1000)
    model.fit(x_tr, y_tr)
    y_pred = model.predict(x_val)
    results.append({
        'C': C,
        'Accuracy': accuracy_score(y_val, y_pred),
        'Precision': precision_score(y_val, y_pred),
        'Recall': recall_score(y_val, y_pred),
        'F1 Score': f1_score(y_val, y_pred)
    })
   
results

[{'C': 0.01,
  'Accuracy': 0.5082927798192471,
  'Precision': 0.3073083067092652,
  'Recall': 0.5094339622641509,
  'F1 Score': 0.3833603188441898},
 {'C': 0.1,
  'Accuracy': 0.5089879829178667,
  'Precision': 0.30719871666332466,
  'Recall': 0.5071168487255876,
  'F1 Score': 0.3826173826173826},
 {'C': 1,
  'Accuracy': 0.5096831860164862,
  'Precision': 0.30724346076458753,
  'Recall': 0.5054617676266137,
  'F1 Score': 0.3821799524465023},
 {'C': 10,
  'Accuracy': 0.5096831860164862,
  'Precision': 0.30724346076458753,
  'Recall': 0.5054617676266137,
  'F1 Score': 0.3821799524465023},
 {'C': 100,
  'Accuracy': 0.5096831860164862,
  'Precision': 0.30724346076458753,
  'Recall': 0.5054617676266137,
  'F1 Score': 0.3821799524465023}]

- C - 0.01 gives the best results.

In [19]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight='balanced',
    random_state=42
)

rf.fit(x_tr, y_tr)
rf_preds = rf.predict(x_val)

rf_results = {
    "Model": "RandomForest",
    "Accuracy": accuracy_score(y_val, rf_preds),
    "Precision": precision_score(y_val, rf_preds),
    "Recall": recall_score(y_val, rf_preds),
    "F1 Score": f1_score(y_val, rf_preds),
}
rf_results


{'Model': 'RandomForest',
 'Accuracy': 0.5121660542258417,
 'Precision': 0.2970594548186306,
 'Recall': 0.4581264481959616,
 'F1 Score': 0.36041666666666666}

In [20]:


results = [
    {"Model": "LogReg (C=0.01)", "Accuracy": 0.5083, "Precision": 0.3073, "Recall": 0.5094, "F1 Score": 0.3834},
    rf_results,
]

pd.DataFrame(results)


,Model,Accuracy,Precision,Recall,F1 Score
0,LogReg (C=0.01),0.508300,0.307300,0.509400,0.383400
1,RandomForest,0.512166,0.297059,0.458126,0.360417


- given the project being a risk prediction task, recall and F1 score are more important than raw accuracy so the tuned logistic regression is currently the better model.

In [21]:
# another threshold tuning for logistic regression to improve F1 score and recall

import numpy as np

# Predicted probabilities for the positive class
y_proba = model.predict_proba(x_val)[:, 1]

thresholds = [0.5, 0.4, 0.35, 0.3, 0.25]
results = []

for thr in thresholds:
    y_pred_thr = (y_proba >= thr).astype(int)
    
    acc = accuracy_score(y_val, y_pred_thr)
    prec = precision_score(y_val, y_pred_thr)
    rec = recall_score(y_val, y_pred_thr)
    f1 = f1_score(y_val, y_pred_thr)
    
    results.append({
        "Threshold": thr,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    })


pd.DataFrame(results)


,Threshold,Accuracy,Precision,Recall,F1
0,0.50,0.509683,0.307243,0.505462,0.382180
1,0.40,0.300030,0.300030,1.000000,0.461574
2,0.35,0.300030,0.300030,1.000000,0.461574
3,0.30,0.300030,0.300030,1.000000,0.461574
4,0.25,0.300030,0.300030,1.000000,0.461574


- 0.4 - 0.25 give a higher recall meaning they catch all the at-risk cases in validation. 
- lower accuracy, which is a okay since the main goal is not to miss risky students.


In [28]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

ros = RandomOverSampler(random_state=42)
X_tr_res, y_tr_res = ros.fit_resample(x_tr, y_tr)

log_res = LogisticRegression(max_iter=1000)
log_res.fit(X_tr_res, y_tr_res)

probs = log_res.predict_proba(x_val)[:, 1]

thresholds = [0.5, 0.4, 0.35, 0.3]
rows = []
for thr in thresholds:
    preds = (probs >= thr).astype(int)
    rows.append({
        "Threshold": thr,
        "Accuracy": accuracy_score(y_val, preds),
        "Precision": precision_score(y_val, preds),
        "Recall": recall_score(y_val, preds),
        "F1": f1_score(y_val, preds),
    })

pd.DataFrame(rows)


/Users/tg197682/anaconda3/lib/python3.11/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/Users/tg197682/anaconda3/lib/python3.11/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


,Threshold,Accuracy,Precision,Recall,F1
0,0.50,0.505512,0.306062,0.51142,0.382947
1,0.40,0.300030,0.300030,1.00000,0.461574
2,0.35,0.300030,0.300030,1.00000,0.461574
3,0.30,0.300030,0.300030,1.00000,0.461574


In [26]:
# Refiting on full training set
x_full = df_train.drop('Addiction_Class', axis=1)
y_full = df_train['Addiction_Class']
model = LogisticRegression(class_weight='balanced', max_iter=1000, C=0.01)
model.fit(x_full, y_full)

# Testing evaluation with chosen threshold
x_test = df_test.drop('Addiction_Class', axis=1)
y_test = df_test['Addiction_Class']

y_test_proba = model.predict_proba(x_test)[:, 1]
thr = 0.35  # chosen threshold from validation
y_test_pred = (y_test_proba >= thr).astype(int)

print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred))
print("Recall:", recall_score(y_test, y_test_pred))
print("F1:", f1_score(y_test, y_test_pred))


Accuracy: 0.20707784055241682
Precision: 0.20707784055241682
Recall: 1.0
F1: 0.34310602613274394


- small number of binary features and weak correlations limit performance.
